# 10주차 실습 — 안전한 데이터 수정과 트랜잭션

- **데이터**: 메모리 안의 빈 연습용 데이터베이스 `emr`. 7주차에 읽어 본 `CREATE TABLE 환자` (슬라이드 166)로 표를 직접 만들어 씁니다. 원본 `aact_mini` 는 건드리지 않습니다.
- **오늘 배우는 것**
  1. INSERT · UPDATE · DELETE의 기본 모양을 읽고 쓸 수 있다
  2. WHERE를 빠뜨렸을 때 무슨 일이 생기는지 설명할 수 있다
  3. 트랜잭션이 왜 필요한지 병원 예로 설명할 수 있다
  4. 안전한 수정 절차 여섯 단계를 말할 수 있다
- **사용법**: 따라 하기(1~7절) → 직접 해보기 6문항. 표가 꼬이면 언제든 `처음상태로()` 를 실행하면 1 · 2절을 마친 처음 상태(7명)로 돌아갑니다.
- **점수는 없습니다.** 마음껏 바꾸고 지워 보세요.

## 0. 시작하기 전에

- **Colab 에서 열었다면 먼저 메뉴의 `파일 → Drive에 사본 저장`** 을 누르세요. 그래야 작성한 내용이 남습니다.
- 셀 실행: 셀 왼쪽의 ▶ 단추 또는 `Shift + Enter`. **위에서부터 차례로** 실행합니다.
- `%%sql` 로 시작하는 셀에는 SQL만 적습니다. `%%sql` 바로 아래의 `-- Q1` 같은 줄은 지우지 마세요 (문항 이름표입니다).
- 오류가 나면 붉은 글씨 중 **`Error:` 가 들어간 줄**만 읽으면 됩니다. 연결이 이상하면 `런타임 → 런타임 다시 시작` 후 준비 셀부터 다시 실행하세요.

In [ ]:
#@title ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게
%sql conn --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    _LAST.update(tag=tag, error=not result.success)

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 학번이 입력되지 않아 배정 질환이 없습니다. 맨 위 학번 셀과 준비 셀을 다시 실행하세요."); return
    if _LAST["error"]:
        return _miss(name, "바로 위 SQL 셀에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 SQL을 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, f"결과가 표가 아닙니다. %%sql 셀을 실행한 뒤, 바로 다음 셀에서 check('{name}', _) 를 실행하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 check 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

In [ ]:
#@title 채점용 정보 (수정하지 마세요) { display-mode: "form" }
ANSWERS = {'Q1': {'sig': '7ea607a1cfd8', 'osig': '7ea607a1cfd8', 'rows': 8, 'cols': ['환자번호', '이름', '생년월일', '성별', '진료과'], 'hint': '결과 확인 표에 107번 한가람이 새로 보여야 합니다(모두 8명). 값은 괄호 안 열의 순서(환자번호, 이름, 생년월일)대로 적고, 숫자는 그대로, 문자와 날짜는 작은따옴표로 감쌉니다. 수정은 실행할 때마다 표를 바꾸므로, 다시 할 때는 이 문항의 처음상태로() 셀부터 실행하세요.', 'ordered': False, 'optional': False}, 'Q2': {'sig': 'dd41b8971507', 'osig': '2e9340c67ffa', 'rows': 6, 'cols': ['column_name', 'data_type'], 'hint': '결과 확인 표 맨 아래에 체중 · INTEGER 한 줄이 새로 보여야 합니다. ADD 뒤에 새 열 이름, 그다음 타입을 문항에 적힌 그대로 적습니다. 수정은 실행할 때마다 표를 바꾸므로, 다시 할 때는 이 문항의 처음상태로() 셀부터 실행하세요.', 'ordered': False, 'optional': False}, 'Q3': {'sig': 'f9196e5bcb2a', 'osig': 'f9196e5bcb2a', 'rows': 1, 'cols': ['환자수'], 'hint': "WHERE 에 '진료과가 빈 값' 조건을 적습니다. 빈 값은 = 로 비교할 수 없습니다 (7주차). 빈 값을 찾는 방법을 떠올리세요.", 'ordered': False, 'optional': False}, 'Q4': {'sig': 'cd5c9147c13d', 'osig': 'cd5c9147c13d', 'rows': 7, 'cols': ['환자번호', '이름', '생년월일', '성별', '진료과'], 'hint': '결과 확인 표에서 진료과가 비어 있던 환자만 내과로 바뀌어야 합니다. 모두 내과면 WHERE 를 빠뜨린 것, 그대로면 빈 값을 = 로 비교한 것입니다. SET 에는 새 값, WHERE 에는 Q3 과 같은 조건을 적습니다. 수정은 실행할 때마다 표를 바꾸므로, 다시 할 때는 이 문항의 처음상태로() 셀부터 실행하세요.', 'ordered': False, 'optional': False}, 'Q5': {'sig': '2678f8c54721', 'osig': '2678f8c54721', 'rows': 6, 'cols': ['환자번호', '이름', '생년월일', '성별', '진료과'], 'hint': '결과 확인 표에서 106번 한 행만 사라져 6명이 남아야 합니다. 0행이면 WHERE 를 빠뜨려 모두 지워진 것입니다. 이름이 같은 환자는 여럿일 수 있으니, 한 행만 정확히 고르는 열(기본키)로 지웁니다. 수정은 실행할 때마다 표를 바꾸므로, 다시 할 때는 이 문항의 처음상태로() 셀부터 실행하세요.', 'ordered': False, 'optional': False}, 'Q6': {'sig': 'b4731a4fd708', 'osig': 'b4731a4fd708', 'rows': 7, 'cols': ['환자번호', '이름', '생년월일', '성별', '진료과'], 'hint': "결과 확인 표의 진료과가 준비 전(처음 상태)으로 돌아와야 합니다. 확정(COMMIT)이 아니라, BEGIN 이후의 변경을 없던 일로 하는 명령입니다. COMMIT 을 했거나 'no transaction is active' 오류가 나면 Q6 준비 셀부터 다시 실행하세요.", 'ordered': False, 'optional': False}}

### 연습용 데이터베이스 만들기

지금까지 배운 SELECT는 데이터를 읽기만 했습니다. 오늘은 데이터를 **바꿉니다.** 수정은 데이터가 실제로 바뀌고, 조건을 잘못 쓰면 전체가 바뀝니다 (슬라이드 267).
원본 `aact_mini` 는 읽기 전용이므로, 아래 셀이 메모리 안에 **빈 연습용 데이터베이스(emr)** 를 만듭니다. 연습은 복사본에서 합니다 (슬라이드 280).
중간에 런타임을 다시 시작했거나 준비 셀을 다시 실행했다면, 이 셀을 다시 실행하고 이어서 `처음상태로()` 를 실행하세요 (환자 표가 7명 상태로 다시 만들어집니다).

In [ ]:
#@title ▶ 연습용 데이터베이스 만들기 — 실행만 하세요 (Shift+Enter) { display-mode: "form" }
class _한연결:
    """%%sql 셀이 늘 같은 연결을 쓰게 합니다. 그래야 BEGIN 이 다음 셀의 COMMIT · ROLLBACK 까지 이어집니다 (5절)."""
    def __init__(self, con): self.con = con
    def cursor(self): return self
    def commit(self): pass        # 확정은 SQL 의 COMMIT; 으로. BEGIN 없이 실행한 문장은 바로 확정(auto-commit)
    def close(self): pass
    def __getattr__(self, name): return getattr(self.con, name)

if "emr" not in globals():
    emr = _한연결(duckdb.connect())          # 메모리 안의 빈 연습용 데이터베이스 (읽기 · 쓰기 가능)
    emr.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게 (준비 셀과 같음)
    emr.execute("CREATE TABLE 진료과 (코드 VARCHAR(10) PRIMARY KEY)")   # 환자 표의 REFERENCES 진료과(코드) 가 가리킬 표
    emr.execute("INSERT INTO 진료과 VALUES ('내과'), ('외과'), ('소아과')")
%sql emr --alias emr

def 처음상태로():
    """환자 표를 1 · 2절을 마친 처음 상태(7명)로 되돌립니다."""
    try:
        emr.execute("ROLLBACK")              # 끝나지 않은 트랜잭션(BEGIN)이 있으면 정리
    except Exception:
        pass
    emr.execute("DROP VIEW IF EXISTS 내과_환자")
    emr.execute("DROP TABLE IF EXISTS 환자")
    emr.execute("""CREATE TABLE 환자 (
    환자번호  INT          PRIMARY KEY,
    이름      VARCHAR(20)  NOT NULL,
    생년월일  DATE,
    성별      CHAR(1)      CHECK (성별 IN ('M','F')),
    진료과    VARCHAR(10)  REFERENCES 진료과(코드)
);""")
    emr.execute("""INSERT INTO 환자
  (환자번호, 이름, 생년월일)
VALUES (100, '김OO', '1958-03-11');""")
    emr.execute("""INSERT INTO 환자
  (환자번호, 이름, 생년월일, 성별, 진료과)
VALUES
  (101, '이OO', '1955-07-30', 'F', '외과'),
  (102, '박OO', '1962-11-05', 'M', '외과'),
  (103, '최OO', '1971-04-22', 'F', '내과'),
  (104, '정OO', '2016-09-09', 'M', '소아과'),
  (105, '강OO', '1983-01-17', 'F', NULL),
  (106, '박OO', '1962-11-05', 'M', '외과');""")
    get_ipython().run_line_magic("sql", "emr --alias emr")
    print("환자 표를 처음 상태(7명)로 되돌렸습니다.")

print("연습용 데이터베이스 emr 에 연결했습니다. 진료과 표의 코드: 내과 · 외과 · 소아과")

## 1. CREATE TABLE — 표 만들기 (7주차 슬라이드 166 복습)

“환자 표를 만들어 줘” — 7주차에 읽어 본 SQL을 이번에는 직접 실행합니다.

| 규칙 | 뜻 |
|---|---|
| `PRIMARY KEY` | 기본키. 겹치거나 비면 안 됨 |
| `NOT NULL` | 빈 값(NULL) 금지 |
| `CHECK` | 정해진 값만 허용 |
| `REFERENCES` | 외래키. 다른 표에 있는 값만 |

`REFERENCES 진료과(코드)` 가 가리키는 진료과 표(코드: 내과 · 외과 · 소아과)는 위 셀이 미리 만들어 두었습니다.

In [ ]:
%%sql
-- 표 만들기 → 열 이름만 있고 행은 아직 없습니다
CREATE TABLE 환자 (
    환자번호  INT          PRIMARY KEY,
    이름      VARCHAR(20)  NOT NULL,
    생년월일  DATE,
    성별      CHAR(1)      CHECK (성별 IN ('M','F')),
    진료과    VARCHAR(10)  REFERENCES 진료과(코드)
);

SELECT * FROM 환자;

표를 만드는 문장은 결과가 없어서, 셀 끝에 `SELECT` 를 붙여 확인합니다.
이 셀을 두 번 실행하면 `already exists`(이미 있음) 오류가 납니다. 표는 이미 만들어졌으니 그대로 다음으로 넘어가면 됩니다.

## 2. INSERT — 새 행 추가

열 이름과 값을 **같은 순서로** 적습니다. 문자와 날짜는 '작은따옴표', 숫자는 따옴표 없이 씁니다.

In [ ]:
%%sql
-- INSERT: 새 행을 추가 → 넣지 않은 성별 · 진료과는 빈 값(None)
INSERT INTO 환자
  (환자번호, 이름, 생년월일)
VALUES (100, '김OO', '1958-03-11');

SELECT * FROM 환자 ORDER BY 환자번호;

여러 행은 괄호 묶음을 쉼표로 이어 한 번에 넣습니다. 모르는 값은 `NULL`(빈 값)로 둡니다.

In [ ]:
%%sql
-- 여러 행을 한 번에 → 모두 7명
INSERT INTO 환자
  (환자번호, 이름, 생년월일, 성별, 진료과)
VALUES
  (101, '이OO', '1955-07-30', 'F', '외과'),
  (102, '박OO', '1962-11-05', 'M', '외과'),
  (103, '최OO', '1971-04-22', 'F', '내과'),
  (104, '정OO', '2016-09-09', 'M', '소아과'),
  (105, '강OO', '1983-01-17', 'F', NULL),
  (106, '박OO', '1962-11-05', 'M', '외과');

SELECT * FROM 환자 ORDER BY 환자번호;

### 규칙에 어긋나는 값은 거부됩니다

INSERT 할 때 확인할 것: 기본키 중복, 필수 열 누락, 타입 (슬라이드 271). 1절의 규칙이 여기서 실제로 작동합니다. 5주차의 무결성 제약조건입니다.

In [ ]:
%%sql
-- [오류 예시] 같은 환자번호(100)를 또 넣으면 → Duplicate key ... primary key (기본키 중복)
INSERT INTO 환자
  (환자번호, 이름, 생년월일)
VALUES (100, '김OO', '1958-03-11');

같은 INSERT 셀을 두 번 실행해도 이 오류가 납니다. 이미 들어가 있으니 그대로 넘어가면 됩니다.

## 3. ALTER TABLE — 표 고치기 (7주차 슬라이드 167 복습)

표의 구조(열)를 바꿉니다. 7주차에 읽어 본 SQL을 직접 실행해 봅니다.

| 하고 싶은 일 | SQL | 결과 |
|---|---|---|
| 열 추가 | `ALTER TABLE 환자 ADD 키 INT;` | 키 열이 새로 생긴다 |
| 열 크기 변경 | `ALTER TABLE 환자 ALTER 이름 TYPE VARCHAR(30);` | 이름을 30자까지 담는다 |
| 열 이름 변경 | `ALTER TABLE 환자 RENAME COLUMN 환자번호 TO ID;` | 열 이름만 바뀐다 |
| 열 삭제 | `ALTER TABLE 환자 DROP COLUMN 키;` | 키 열과 그 안의 값이 사라진다 |

In [ ]:
%%sql
-- 열 추가 → 키 열이 새로 생긴다 (값은 아직 빈 값)
ALTER TABLE 환자 ADD 키 INT;

SELECT * FROM 환자;

In [ ]:
%%sql
-- 열 크기 변경 → 화면은 같지만 이름을 30자까지 담는다
ALTER TABLE 환자 ALTER 이름 TYPE VARCHAR(30);

SELECT * FROM 환자;

In [ ]:
%%sql
-- 열 이름 변경 → 환자번호 가 ID 로 바뀐다
ALTER TABLE 환자 RENAME COLUMN 환자번호 TO ID;

SELECT * FROM 환자;

> 참고: 슬라이드의 `DROP COLUMN 생년월일` 은 이 표에서 DuckDB 오류가 나서, 방금 추가한 키 열로 해 봅니다.

In [ ]:
%%sql
-- 열 삭제 → 키 열과 그 안의 값이 사라진다
ALTER TABLE 환자 DROP COLUMN 키;

SELECT * FROM 환자;

### 처음 상태로 되돌리기

ALTER 로 표 구조가 바뀌었습니다. 아래 셀을 실행해 1 · 2절을 마친 **처음 상태(7명)** 로 되돌립니다.
앞으로도 표가 꼬이면 언제든 이 셀을 실행하세요.

In [ ]:
처음상태로()

## 4. UPDATE · DELETE — WHERE 와 SELECT 먼저

| 문장 | 하는 일 |
|---|---|
| `INSERT` | 새 행을 추가 |
| `UPDATE` | 기존 값을 바꿈 |
| `DELETE` | 행을 지움 |

**UPDATE와 DELETE에는 WHERE가 거의 반드시 필요합니다.** 빠뜨리면 모든 행이 대상이 됩니다 (슬라이드 269).

In [ ]:
%%sql
-- UPDATE: 기존 값을 바꿈 → 100번 환자의 진료과만 바뀜
UPDATE 환자
SET 진료과 = '내과'
WHERE 환자번호 = 100;

SELECT * FROM 환자 ORDER BY 환자번호;

In [ ]:
%%sql
-- DELETE: 행을 지움 → 100번 환자만 사라짐
DELETE FROM 환자
WHERE 환자번호 = 100;

SELECT * FROM 환자 ORDER BY 환자번호;

### WHERE를 빠뜨리면 — 오류가 나지 않아서 더 위험합니다

실무에서 가장 유명한 사고입니다. 오류가 나지 않고 **조용히 전부 바뀝니다** (슬라이드 270).

In [ ]:
%%sql
-- [오류 예시] WHERE 를 빠뜨림 → 오류 없이 모든 환자의 진료과가 내과로 바뀜
UPDATE 환자
SET 진료과 = '내과';
-- WHERE 가 없습니다

SELECT * FROM 환자 ORDER BY 환자번호;

→ **모든 환자**의 진료과가 내과로 바뀌었습니다. 5만 명이면 5만 행이 한 번에 바뀝니다. 연습용 표이므로 처음 상태로 되돌립니다.

In [ ]:
처음상태로()

### 이렇게 확인하고 씁니다 — SELECT 먼저

바꿀 조건을 WHERE에 적고, **같은 조건으로 먼저 세어 본 뒤** 수정합니다.

In [ ]:
%%sql
-- ① 같은 WHERE 로 먼저 세어 보기 → 3명
SELECT COUNT(*) FROM 환자
WHERE 진료과 = '외과';

In [ ]:
%%sql
-- ② 수정 → 결과의 Count 가 바뀐 행 수. ①에서 센 3과 같은지 확인
UPDATE 환자
SET 진료과 = '내과'
WHERE 진료과 = '외과';

> **병원 데이터라면** — 처방 코드 일괄 정정 작업에서 WHERE를 빠뜨려 수개월치 처방이 한 값으로 덮여 쓰인 사고가 있었습니다.
> 그래서 운영 데이터베이스 수정은 혼자 하지 않고, 백업을 확인한 뒤, 승인 절차를 거쳐 진행합니다.

## 5. 트랜잭션 — COMMIT과 ROLLBACK

**트랜잭션** = 여러 개의 수정을 하나의 덩어리로 묶은 것. 전부 되거나, 전부 안 되거나 (슬라이드 274).

> **병원 데이터라면** — 입원 수속은 ① 입원 기록을 만든다 ② 병상을 ‘사용 중’으로 바꾼다 ③ 수납 내역을 만든다 가 한 덩어리입니다.
> ②에서 오류가 나면 입원 기록은 있는데 병상은 비어 있는 상태가 됩니다. 그래서 묶어서, 하나라도 실패하면 전부 없던 일로 되돌립니다.

| 명령 | 하는 일 |
|---|---|
| `BEGIN;` | 여기부터 한 덩어리(트랜잭션)로 묶기 시작 |
| `COMMIT;` | 결과가 맞으면: 바뀐 내용을 확정합니다. 이제 되돌릴 수 없습니다. |
| `ROLLBACK;` | 이상하면: BEGIN 이후의 모든 변경을 없던 일로 되돌립니다. |

ACID — 트랜잭션이 지켜 주는 네 가지 (용어만 기억하면 됩니다): 원자성(전부 또는 전무) · 일관성(규칙 유지) · 고립성(서로 방해 안 함) · 지속성(확정되면 남음)

### 결과가 맞으면 COMMIT

BEGIN 이 든 셀은 **한 번만** 실행하세요. 두 번 실행해 `cannot start a transaction within a transaction` 오류가 나면, `처음상태로()` 를 실행한 뒤 BEGIN 셀부터 다시 하면 됩니다.

In [ ]:
%%sql
-- BEGIN 후 수정하고, 결과를 조회해 확인 (아직 확정 전)
BEGIN;

UPDATE 환자
SET 진료과 = '내과'
WHERE 환자번호 = 100;

SELECT * FROM 환자 WHERE 환자번호 = 100;

In [ ]:
%%sql
-- 결과가 맞으면 COMMIT → 바뀐 내용을 확정
COMMIT;

SELECT * FROM 환자 WHERE 환자번호 = 100;

### 이상하면 ROLLBACK

In [ ]:
%%sql
-- [오류 예시] BEGIN 뒤에 WHERE 를 빠뜨린 DELETE → 0행 (아직 확정 전)
BEGIN;

DELETE FROM 환자;
-- WHERE 를 빠뜨렸습니다

SELECT * FROM 환자 ORDER BY 환자번호;

In [ ]:
%%sql
-- 이상하면 ROLLBACK → BEGIN 이후의 변경이 없던 일이 되어 7명이 그대로
ROLLBACK;

SELECT * FROM 환자 ORDER BY 환자번호;

**중요**: COMMIT 하기 전에는 다른 사람에게 이 변경이 보이지 않습니다 (고립성).
일부 도구는 자동으로 COMMIT 됩니다(auto-commit). 이 노트북도 BEGIN 없이 실행한 수정은 바로 확정되어 ROLLBACK 으로 되돌릴 수 없습니다. 4절에서 WHERE 를 빠뜨린 UPDATE 를 `처음상태로()` 로 되돌린 이유입니다.

### 안전한 수정 절차 — 실무에서 지키는 여섯 단계 (슬라이드 279)

| 단계 | 할 일 |
|---|---|
| ① 환경 확인 | 운영 데이터베이스인가, 연습용 복사본인가 |
| ② 백업 확인 | 언제 백업되었는지, 복구에 얼마나 걸리는지 |
| ③ SELECT 먼저 | 같은 WHERE로 조회해 바뀔 행 수를 확인 |
| ④ BEGIN 후 수정 | 트랜잭션 안에서 UPDATE · DELETE 실행 |
| ⑤ 결과 확인 | 바뀐 행 수와 실제 값을 다시 조회 |
| ⑥ COMMIT / ROLLBACK | 맞으면 확정, 이상하면 되돌리기 |

이 순서를 지키면 사고의 대부분을 막을 수 있습니다.

## 6. 뷰(View) — 자주 쓰는 조회를 저장하기

“이 조건을 매번 다시 쓰지 않게 해 주세요” — 긴 조건을 이름 하나로 줄여 둡니다 (슬라이드 281). 여기서는 연습용 환자 표로 만듭니다.
뷰는 표를 새로 만드는 것이 아니라 **조회 방법에 이름을 붙이는 것**입니다. 원본은 그대로 남습니다.

In [ ]:
%%sql
-- 뷰 만들기 → 이름으로 세기
CREATE VIEW 내과_환자 AS
SELECT 환자번호, 생년월일, 성별
FROM 환자
WHERE 진료과 = '내과';

SELECT COUNT(*) FROM 내과_환자;

In [ ]:
%%sql
-- 뷰도 표처럼 조회합니다 → 이름 열은 보이지 않습니다
SELECT * FROM 내과_환자 ORDER BY 환자번호;

> **병원 데이터라면** — 이름 · 연락처를 뺀 뷰로 권한을 제한할 수 있습니다. 분석자에게는 대개 뷰를 만들 권한이 없으니, 필요하면 데이터 담당 부서에 요청합니다 (슬라이드 282).

## 7. DROP TABLE — 표 지우기 (7주차 슬라이드 167 복습)

DROP은 데이터까지 지우고 되돌릴 수 없어 관리자만 씁니다. 연습용 표이므로 지워 봅니다.

In [ ]:
%%sql
-- 표 전체가 사라진다
DROP TABLE 환자;

In [ ]:
%%sql
-- [오류 예시] 지운 표를 조회하면 → Table with name 환자 does not exist!
SELECT * FROM 환자 ORDER BY 환자번호;

오류 메시지 `Table with name 환자 does not exist!` = 환자라는 표가 없다. **표가 사라졌습니다.** 그 안의 7명도 함께 사라졌습니다.
아래 직접 해보기는 `처음상태로()` 로 표를 다시 만들고 시작합니다.

## 직접 해보기 — 6문항

- **각 문항은 처음 상태(7명)에서 시작합니다.** 순서: ① `처음상태로()` 셀 → ② `%%sql` 셀의 `___` 를 채워 실행 → ③ `check(...)` 셀. (Q6 은 ① 이 준비 셀입니다.)
- `%%sql` 셀 아래쪽의 `결과 확인` SELECT 는 바뀐 표를 다시 보는 부분입니다. 지우지 마세요.
- ❌ 가 나오면 `결과 확인` 표에서 **바꾸려던 행만 바뀌었는지** 보세요. 두 번째 ❌ 부터는 그 문항의 힌트가 나옵니다.
- 수정은 실행할 때마다 표를 바꿉니다. 고쳐서 다시 할 때나, 같은 셀을 두 번 실행해 오류(`Duplicate key` · `already exists`)가 났을 때는 그 문항의 ① 셀부터 다시 실행하세요.

**Q1.** 새 환자를 등록하세요 — 환자번호 `107`, 이름 `'한가람'`, 생년월일 `'1990-05-20'`. (INSERT, 2절)

In [ ]:
처음상태로()

In [ ]:
%%sql
-- Q1
INSERT INTO 환자 (환자번호, 이름, 생년월일)
VALUES (___, ___, ___);

-- 결과 확인 (이 줄은 그대로 두세요)
SELECT * FROM 환자 ORDER BY 환자번호;

In [ ]:
check("Q1", _)

**Q2.** 환자 표에 체중을 적을 열 `체중` 을 정수(`INT`)로 추가하세요. 이번 `결과 확인` 은 7주차의 데이터 사전(`information_schema`)으로 환자 표의 열 이름과 타입을 보여 줍니다 (`INT` 는 `INTEGER` 로 보입니다). (ALTER TABLE, 3절)

In [ ]:
처음상태로()

In [ ]:
%%sql
-- Q2
ALTER TABLE 환자 ADD ___ ___;

-- 결과 확인: 환자 표의 열 이름과 타입 (아래는 그대로 두세요)
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = '환자'
ORDER BY ordinal_position;

In [ ]:
check("Q2", _)

**Q3.** 진료과를 비워 두고(NULL) 등록한 환자가 있습니다. 진료과를 채우기 전에, **진료과가 빈 값(NULL)인 환자가 몇 명인지 먼저** 세어 보세요. 다음 Q4 에서 이 WHERE 를 그대로 씁니다. (SELECT 먼저, 4절)

In [ ]:
처음상태로()

In [ ]:
%%sql
-- Q3
SELECT COUNT(*) AS 환자수
FROM 환자
WHERE ___;

In [ ]:
check("Q3", _)

**Q4.** 확인해 보니 진료과가 비어 있던 환자는 모두 내과 환자였습니다. Q3 과 **같은 WHERE** 로 진료과를 `'내과'` 로 채우세요. (UPDATE, 4절)

In [ ]:
처음상태로()

In [ ]:
%%sql
-- Q4
UPDATE 환자
SET 진료과 = ___
WHERE ___;

-- 결과 확인 (이 줄은 그대로 두세요)
SELECT * FROM 환자 ORDER BY 환자번호;

In [ ]:
check("Q4", _)

**Q5.** 106번은 102번과 같은 사람(이름 · 생년월일 · 성별 · 진료과가 같음)이 두 번 등록된 것입니다. 중복으로 등록된 **106번 한 행만** 지우세요. (DELETE, 4절)

In [ ]:
처음상태로()

In [ ]:
%%sql
-- Q5
DELETE FROM 환자
WHERE ___;

-- 결과 확인 (이 줄은 그대로 두세요)
SELECT * FROM 환자 ORDER BY 환자번호;

In [ ]:
check("Q5", _)

**Q6.** 아래 준비 셀은 처음 상태로 되돌린 뒤 BEGIN 하고, **WHERE 를 빠뜨린 UPDATE** 를 실행합니다 (모든 환자가 소아과, 아직 확정 전). Q6 셀의 `___` 에 알맞은 명령을 넣어, BEGIN 이후의 변경을 없던 일로 되돌리세요. (ROLLBACK, 5절)

In [ ]:
# Q6 준비 — 처음 상태로 되돌린 뒤 BEGIN 하고, WHERE 를 빠뜨린 UPDATE 를 실행합니다 (여러 번 실행해도 됩니다)
처음상태로()
%sql BEGIN;
%sql UPDATE 환자 SET 진료과 = '소아과';
%sql SELECT * FROM 환자 ORDER BY 환자번호;

In [ ]:
%%sql
-- Q6
___;

-- 결과 확인 (이 줄은 그대로 두세요)
SELECT * FROM 환자 ORDER BY 환자번호;

In [ ]:
check("Q6", _)

## 오늘 배운 것

- **UPDATE · DELETE 에는 WHERE.** 빠뜨리면 모든 행이 바뀐다
- **같은 WHERE로 SELECT 먼저** 해서 바뀔 행 수를 확인한다
- 여러 수정은 **트랜잭션**으로 묶고, 확인 후 **COMMIT 또는 ROLLBACK**

다음 시간: 11강. CDW 구조와 분석 업무 흐름 — 병원에서 데이터를 받는 과정

아래 셀을 실행하면 6문항의 결과를 한눈에 볼 수 있습니다.

In [ ]:
진행확인()